# Investment Portfolio Risk Analytics

**Research question:** Does walk-forward mean-variance optimization improve risk-adjusted performance over a diversified ETF portfolio after transaction costs?

**ETFs:** SPY, EFA, EEM, AGG, GLD. **Base weights:** 40%, 20%, 10%, 20%, 10%.  
**Method:** Daily adjusted USD ETF prices, historical risk analysis, and monthly rebalanced walk-forward testing with a 504-trading-day lookback.

Run this notebook top to bottom. Figures depend on the data provider and the date of download. This is an educational backtest, not investment advice.

## 1. Setup and historical prices

In [ ]:
# In Google Colab, uncomment if packages are missing:
# %pip install -q yfinance pandas numpy scipy matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from scipy.optimize import minimize

TICKERS = ["SPY", "EFA", "EEM", "AGG", "GLD"]
START = "2021-01-01"
END = "2026-11-19"  # exclusive: fixed end date for a reproducible data window
TRADING_DAYS = 252
RISK_FREE_RATE = 0.03  # illustrative constant, not historical daily cash returns
COST_RATE = 0.001  # 10 bps of traded notional, illustrative
LOOKBACK = 504
INITIAL_CAPITAL = 10_000.0

base_weights = pd.Series({"SPY": .40, "EFA": .20, "EEM": .10, "AGG": .20, "GLD": .10}).reindex(TICKERS)
raw = yf.download(TICKERS, start=START, end=END, auto_adjust=True, progress=False)
prices = raw["Close"].reindex(columns=TICKERS).dropna(how="any")
if len(prices) < LOOKBACK + 60:
    raise ValueError("Insufficient complete observations for walk-forward testing")
returns = prices.pct_change().dropna()
assert returns.index.is_monotonic_increasing
print(f"Data: {prices.index.min().date()} to {prices.index.max().date()}, {len(returns)} returns")
display(prices.tail())

### Data and interpretation
`auto_adjust=True` adjusts historical prices for splits and distributions according to the data provider. The resulting return series approximates total return in USD. Rows with a missing ETF price are dropped to ensure a shared calendar. Historical adjusted prices can be revised by the provider.

## 2. Historical performance and risk

In [ ]:
def summary_stats(daily_returns, rf=RISK_FREE_RATE):
    r = daily_returns.dropna()
    compound = (1 + r).prod() ** (TRADING_DAYS / len(r)) - 1
    vol = r.std(ddof=1) * np.sqrt(TRADING_DAYS)
    wealth = pd.concat([pd.Series([1.0]), (1+r).cumprod().reset_index(drop=True)], ignore_index=True)
    drawdown = (wealth / wealth.cummax() - 1).min()
    var = -r.quantile(.05)
    es = -r[r <= r.quantile(.05)].mean()
    return pd.Series({"Annual return":compound, "Volatility":vol,
                      "Sharpe ratio":(compound-rf)/vol,
                      "Maximum drawdown":drawdown,
                      "1d historical VaR 95%":var, "1d ES 95%":es})

# Constant mix: portfolio weights restored daily. Descriptive only, no trading costs.
base_daily = returns @ base_weights
historical = pd.DataFrame({"Original (daily constant mix)":summary_stats(base_daily),
                           "SPY":summary_stats(returns["SPY"])})
display(historical.round(4))

corr = returns.corr()
display(corr.round(2))
plt.figure(figsize=(7, 5))
plt.imshow(corr, vmin=-1, vmax=1, cmap="RdBu_r")
plt.colorbar(label="Daily return correlation")
plt.xticks(range(len(TICKERS)), TICKERS)
plt.yticks(range(len(TICKERS)), TICKERS)
for i in range(len(TICKERS)):
    for j in range(len(TICKERS)):
        plt.text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center")
plt.title("ETF Correlations")
plt.tight_layout()
from pathlib import Path
Path("figures").mkdir(exist_ok=True)
plt.savefig("figures/correlation_matrix_current_run.png", dpi=170, bbox_inches="tight")
plt.show()

## 3. Risk contribution of the original weights

In [ ]:
cov_all = returns.cov() * TRADING_DAYS
variance = float(base_weights @ cov_all @ base_weights)
contrib = base_weights * (cov_all @ base_weights) / variance
print("Percentage contribution to portfolio variance:")
display((100 * contrib).sort_values(ascending=False).round(2))
assert np.isclose(contrib.sum(), 1.0)

## 4. Walk-forward optimization

At the **first trading day of each calendar month**, optimize weights using *only* the preceding 504 daily returns. Trade at that day's start, then apply that day's close-to-close return. No short sales; each ETF is capped at 60%. The original strategy rebalances to its fixed allocation on the same dates.

Costs are 10 bps of traded notional, and assets drift between rebalances. The cost calculation uses a small approximation: it computes turnover on pre-cost wealth, deducts costs, then allocates remaining wealth at the target proportions. Both strategies use exactly the same convention.

In [ ]:
def optimize_weights(history):
    mu = history.mean().to_numpy() * TRADING_DAYS
    cov = history.cov().to_numpy() * TRADING_DAYS
    n = len(history.columns)

    def negative_sharpe(w):
        sigma = np.sqrt(max(float(w @ cov @ w), 1e-16))
        return -(float(w @ mu) - RISK_FREE_RATE) / sigma

    result = minimize(negative_sharpe, np.ones(n)/n, method="SLSQP",
                      bounds=[(0., .60)] * n,
                      constraints=[{"type":"eq", "fun":lambda w: w.sum()-1}],
                      options={"maxiter":1000, "ftol":1e-10})
    if not result.success or abs(result.x.sum()-1) > 1e-6:
        raise RuntimeError(f"Optimization failed: {result.message}")
    return pd.Series(result.x, index=history.columns)

month_starts = returns.index[~returns.index.to_period("M").duplicated()]
rebalance_dates = [d for d in month_starts if returns.index.get_loc(d) >= LOOKBACK]
if not rebalance_dates:
    raise ValueError("No months after the lookback period")
optimized_targets = pd.DataFrame({
    date: optimize_weights(returns.iloc[returns.index.get_loc(date)-LOOKBACK:
                                        returns.index.get_loc(date)])
    for date in rebalance_dates
}).T.reindex(columns=TICKERS)
original_targets = pd.DataFrame([base_weights.to_numpy()] * len(rebalance_dates),
                                index=rebalance_dates, columns=TICKERS)
benchmark_target = pd.DataFrame([{t: float(t=="SPY") for t in TICKERS}],
                                index=[rebalance_dates[0]])
print("Evaluation start:", rebalance_dates[0].date())
print("Average optimized weights (%):")
display((100*optimized_targets.mean()).round(2))

## 5. Simulate monthly holdings, drift and transaction costs

In [ ]:
def run_backtest(daily, targets, initial=INITIAL_CAPITAL, cost_rate=COST_RATE):
    daily = daily.reindex(columns=TICKERS).sort_index()
    targets = targets.reindex(columns=TICKERS).sort_index()
    if targets.isna().any().any() or not np.allclose(targets.sum(axis=1), 1):
        raise ValueError("Invalid target weights")
    if daily.index[0] not in targets.index:
        raise ValueError("A target is required on the first test date")

    holdings = pd.Series(0., index=TICKERS)
    prior_value = initial
    rows = []
    for date, daily_r in daily.iterrows():
        traded = 0.
        fee = 0.
        if date in targets.index:
            target = targets.loc[date]
            current_weight = holdings / holdings.sum() if holdings.sum() > 0 else pd.Series(0., index=TICKERS)
            traded = float((target - current_weight).abs().sum())
            fee = prior_value * traded * cost_rate
            holdings = target * (prior_value - fee)
        holdings = holdings * (1 + daily_r)
        value = float(holdings.sum())
        rows.append((date, value, value/prior_value-1, traded, fee))
        prior_value = value
    return pd.DataFrame(rows, columns=["Date", "Value", "Return", "Turnover", "Cost"]).set_index("Date")

test_returns = returns.loc[rebalance_dates[0]:]
original_bt = run_backtest(test_returns, original_targets)
optimized_bt = run_backtest(test_returns, optimized_targets)
spy_bt = run_backtest(test_returns, benchmark_target)

def backtest_stats(bt):
    res = summary_stats(bt["Return"])
    return pd.concat([res, pd.Series({"Total costs ($)":bt["Cost"].sum(),
                                       "Total turnover":bt["Turnover"].sum(),
                                       "Final wealth ($)":bt["Value"].iloc[-1]})])

comparison = pd.DataFrame({"Original":backtest_stats(original_bt),
                           "Optimized":backtest_stats(optimized_bt),
                           "SPY":backtest_stats(spy_bt)})
print("Test period:", test_returns.index[0].date(), "to", test_returns.index[-1].date())
display(comparison.round(4))
assert original_bt.index.equals(optimized_bt.index)
assert optimized_bt.index.equals(spy_bt.index)
assert (optimized_targets >= -1e-6).all().all() and (optimized_targets <= .600001).all().all()
assert np.allclose(optimized_targets.sum(axis=1), 1)
assert (original_bt[["Value", "Cost", "Turnover"]] >= 0).all().all()

## 6. Results visualization

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(13, 9))
for name, bt in [("Original", original_bt), ("Optimized", optimized_bt), ("SPY", spy_bt)]:
    ax[0,0].plot(bt.index, bt["Value"], label=name)
ax[0,0].set(title="Out-of-sample portfolio values", ylabel="USD")
ax[0,0].legend()
for axis, metric, title, mult in [
    (ax[0,1], "Annual return", "Annualized returns", 100),
    (ax[1,0], "Volatility", "Annualized volatility", 100),
    (ax[1,1], "Maximum drawdown", "Maximum drawdown", 100)]:
    (comparison.loc[metric] * mult).plot(kind="bar", ax=axis)
    axis.set_title(title)
    axis.set_ylabel("%")
    axis.tick_params(axis="x", rotation=0)
for a in ax.flat:
    a.grid(alpha=.2)
plt.suptitle("Portfolio Risk Analytics: Monthly Rebalanced Backtest")
plt.tight_layout()
from pathlib import Path
Path("figures").mkdir(exist_ok=True)
plt.savefig("figures/backtest_dashboard_current_run.png", dpi=170, bbox_inches="tight")
plt.show()

## 7. Interpretation and limitations

- Compare the optimized portfolio with the baseline **on the same dates**, after costs, and account for both return and downside risk. Do not judge solely by the highest annual return.
- **Look-ahead:** optimizer training returns end before each month's first test-day return. However the assumed trading at that day's start based on prior close-to-close returns is an approximation; actual execution prices and market impact are not available.
- **Risk-free rate:** 3% fixed is illustrative. A time-varying USD short-term risk-free return would improve the Sharpe estimate.
- **Expected returns:** sample means are noisy; historical optimization may overfit, and the asset universe is small.
- **Transaction costs:** 10 bps per traded dollar, no taxes, commissions beyond that rate, changing spreads or market impact. Initial allocation turnover is counted.
- **Benchmark:** SPY is an equity-only benchmark, not a risk-matched portfolio.
- **Data:** Yahoo Finance adjusted prices, USD-denominated instruments, provider corrections possible, ETF histories and distributions depend on provider.
- **Backtest:** no investor cash flows, leverage, borrow costs or delistings; asset universe chosen with hindsight. A genuinely investable live strategy requires additional validation.

**Takeaway:** This exercise evaluates whether improved in-sample risk-return estimates translate into better out-of-sample performance. Its outcome is conditional on the dataset and modelling choices, not proof of future alpha.